# KPI Calculations

Formulas for all 12 SaaS KPIs, traffic light thresholds, and health score weighting.

## 1. Load Data and Apply Sample Filter

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "data/processed"

monthly = pd.read_parquet(f"{DATA_PATH}/monthly_metrics.parquet")
monthly["month"] = pd.to_datetime(monthly["month"])
monthly = monthly.sort_values("month")

kpis = pd.read_parquet(f"{DATA_PATH}/monthly_kpis.parquet")
kpis["month"] = pd.to_datetime(kpis["month"])
kpis = kpis.sort_values("month")

# Sample filter: last 12 months, all segments
cutoff = monthly["month"].max() - pd.DateOffset(months=11)
m = monthly[monthly["month"] >= cutoff].copy()

print(f"Loaded {len(m)} months from {m['month'].min().strftime('%Y-%m')} to {m['month'].max().strftime('%Y-%m')}")
print(f"Columns: {list(m.columns)}")

## 2. Revenue KPIs

MRR, ARR, and Net Revenue Retention (NRR) from the Bessemer Venture Partners SaaS framework.

In [ ]:
last = m.iloc[-1]
prev = m.iloc[-2]

# MRR: total contracted subscription revenue recognized in the current month
mrr = last["mrr"]

# ARR: annualized run rate
arr = mrr * 12

# MRR growth rate: month-over-month
mrr_growth_rate = (mrr - prev["mrr"]) / prev["mrr"]

# NRR: (starting_mrr + expansion - contraction - churn) / starting_mrr
# Measures revenue retained and expanded from the existing cohort.
# Above 100% means existing customers alone grow revenue -- the "holy grail" of SaaS.
starting = last.get("starting_mrr", prev["mrr"])
nrr = (
    starting
    + last.get("expansion_mrr", 0)
    - last.get("contraction_mrr", 0)
    - last.get("churned_mrr", 0)
) / starting

print(f"MRR:           ${mrr:>12,.0f}")
print(f"ARR:           ${arr:>12,.0f}")
print(f"MRR Growth:    {mrr_growth_rate * 100:>11.1f}%")
print(f"NRR:           {nrr * 100:>11.1f}%  (target: 110%)")

## 3. Customer KPIs

Logo churn, revenue churn, and NPS.

In [ ]:
# Logo churn rate: count of customers who cancelled / total customers at start of period
logo_churn_rate = last["logo_churn_rate"]

# Revenue churn rate: MRR lost from churned customers / starting MRR
# Revenue churn > logo churn signals that larger accounts are leaving faster -- a critical red flag.
revenue_churn_rate = last["revenue_churn_rate"]

# NPS: % promoters (score 9-10) minus % detractors (score 0-6)
# Ranges from -100 to +100; industry median for B2B SaaS is ~34 (Satmetrix 2024)
nps = last["nps"]

print(f"Logo Churn:     {logo_churn_rate * 100:.2f}%  (target: <2%)")
print(f"Revenue Churn:  {revenue_churn_rate * 100:.2f}%  (target: <1.5%)")
print(f"NPS:            {nps:.0f}            (target: >55)")
print()
if revenue_churn_rate > logo_churn_rate + 0.01:
    print("WARNING: Revenue churn exceeds logo churn -- larger accounts churning at higher rate")

## 4. Efficiency KPIs

LTV, CAC, payback period. These govern unit economics.

In [ ]:
# ARPU: average revenue per user
active_customers = last.get("active_customers", last.get("total_customers", 1))
arpu = mrr / active_customers if active_customers > 0 else 0

# Gross margin: SaaS benchmark 70-80%; used in payback calculation
gross_margin = last.get("gross_margin", 0.75)

# LTV = ARPU / monthly_churn_rate
# Interpretation: average total revenue from a customer over their lifetime
ltv = arpu / logo_churn_rate if logo_churn_rate > 0 else 0

# CAC: total sales + marketing spend / new customers acquired
cac = last.get("cac", 0)

# LTV:CAC ratio: benchmark is >3x; below 1x means growth destroys value
ltv_cac_ratio = ltv / cac if cac > 0 else last.get("ltv_cac_ratio", 0)

# Payback period: months to recover CAC from gross margin contribution
# Benchmark: <12 months for efficient B2B SaaS
payback_months = cac / (arpu * gross_margin) if (arpu * gross_margin) > 0 else last.get("payback_months", 0)

print(f"ARPU:           ${arpu:>8,.0f}/month")
print(f"LTV:            ${ltv:>8,.0f}")
print(f"CAC:            ${cac:>8,.0f}")
print(f"LTV:CAC:        {ltv_cac_ratio:>8.1f}x  (target: >3.5x)")
print(f"Payback:        {payback_months:>8.0f} months  (target: <12)")

## 5. Performance KPIs

Rule of 40, gross margin, and DAU/MAU engagement ratio.

In [ ]:
# Rule of 40: growth rate % + profit margin %
# Originated at Battery Ventures; popularized by Brad Feld.
# Balances growth vs. profitability -- a company growing 20% with 20% margin scores 40.
profit_margin = last.get("profit_margin", last.get("operating_margin", 0))
growth_rate_annual = last.get("arr_growth_rate", mrr_growth_rate * 12)  # annualized
rule_of_40 = (growth_rate_annual * 100) + (profit_margin * 100)

# Gross margin: from the dataset (already computed in generation)
gross_margin_pct = gross_margin * 100

# DAU/MAU: product stickiness; >40% is strong, <20% suggests low engagement
dau_mau = last.get("dau_mau_ratio", 0)

print(f"Rule of 40:     {rule_of_40:>8.1f}  (target: >40)")
print(f"Gross Margin:   {gross_margin_pct:>8.1f}%  (target: >75%)")
print(f"DAU/MAU:        {dau_mau * 100:>8.1f}%  (target: >25%)")

## 6. Traffic Light Logic

Thresholds come from Bessemer Venture Partners State of the Cloud and SaaStr benchmarks for $5-15M ARR B2B SaaS.

In [ ]:
KPI_TARGETS = {
    "mrr":              {"target": 950_000,  "higher_is_better": True},
    "arr":              {"target": 11_400_000, "higher_is_better": True},
    "nrr":              {"target": 1.10,      "higher_is_better": True},   # 110%
    "logo_churn_rate":  {"target": 0.02,      "higher_is_better": False},  # <2%/mo
    "revenue_churn_rate": {"target": 0.015,   "higher_is_better": False},
    "nps":              {"target": 55,        "higher_is_better": True},
    "ltv_cac_ratio":    {"target": 3.5,       "higher_is_better": True},
    "rule_of_40":       {"target": 40,        "higher_is_better": True},
    "gross_margin":     {"target": 0.75,      "higher_is_better": True},
    "dau_mau_ratio":    {"target": 0.25,      "higher_is_better": True},
    "cac":              {"target": 800,       "higher_is_better": False},
    "payback_months":   {"target": 12,        "higher_is_better": False},
}


def traffic_light_status(value: float, target: float, higher_is_better: bool) -> str:
    """green >= 100% of target; yellow >= 90%; red < 90%."""
    if target == 0:
        return "green"
    if higher_is_better:
        ratio = value / target
        if ratio >= 1.0:
            return "green"
        elif ratio >= 0.9:
            return "yellow"
        return "red"
    else:
        # For metrics where lower is better: target acts as a ceiling
        ratio = target / value if value != 0 else 0
        if ratio >= 1.0:
            return "green"
        elif ratio >= 0.9:
            return "yellow"
        return "red"


# Print traffic light for each KPI
kpi_values = {
    "mrr": mrr, "arr": arr, "nrr": nrr,
    "logo_churn_rate": logo_churn_rate, "revenue_churn_rate": revenue_churn_rate,
    "nps": nps, "ltv_cac_ratio": ltv_cac_ratio, "rule_of_40": rule_of_40,
    "gross_margin": gross_margin, "dau_mau_ratio": dau_mau,
    "cac": cac, "payback_months": payback_months,
}

for kpi_name, val in kpi_values.items():
    cfg = KPI_TARGETS[kpi_name]
    status = traffic_light_status(val, cfg["target"], cfg["higher_is_better"])
    print(f"  {kpi_name:<22} {status}")

## 7. Health Score

Composite 0-100 score weighting six dimensions. Equal weights used for v1 to avoid introducing subjective priorities before stakeholder calibration.

In [ ]:
def _scale(value: float, lower: float, upper: float) -> float:
    """Linear scaling: 0.0 at lower, 1.0 at upper, clamped."""
    if upper == lower:
        return 0.5
    return max(0.0, min(1.0, (value - lower) / (upper - lower)))


def compute_health_score(row: dict) -> float:
    """
    Weights:
        MRR growth  20%  | NRR           20%  | Churn       15%
        NPS         15%  | Rule of 40    15%  | LTV:CAC     15%

    Ranges are set to realistic B2B SaaS benchmarks:
        MRR growth: -2% (floor) to +8% (ceiling)
        NRR:        80% to 130%
        Churn:       0% to  8%  (inverted: lower is better)
        NPS:        -20 to 80
        Rule of 40:  10 to 60
        LTV:CAC:    1x  to 6x
    """
    score = 0.0

    mrr_growth = float(row.get("mrr_growth_rate", 0))
    score += _scale(mrr_growth, lower=-0.02, upper=0.08) * 20

    nrr_val = float(row.get("nrr", row.get("net_revenue_retention", 1.0)))
    score += _scale(nrr_val, lower=0.80, upper=1.30) * 20

    churn_val = float(row.get("logo_churn_rate", row.get("churn_rate", 0.02)))
    score += (1.0 - _scale(churn_val, lower=0.0, upper=0.08)) * 15

    nps_val = float(row.get("nps", 50))
    score += _scale(nps_val, lower=-20, upper=80) * 15

    rule40_val = float(row.get("rule_of_40", 30))
    score += _scale(rule40_val, lower=10, upper=60) * 15

    ltv_cac_val = float(row.get("ltv_cac_ratio", 3.0))
    score += _scale(ltv_cac_val, lower=1.0, upper=6.0) * 15

    return round(max(0.0, min(100.0, score)), 1)


# Compute health score for the latest month
metrics_dict = {col: float(last[col]) for col in m.columns if col != "month"}
health = compute_health_score(metrics_dict)

health_status = "green" if health >= 75 else "yellow" if health >= 50 else "red"
print(f"Health Score:  {health:.1f} / 100  ({health_status})")

# Show contribution breakdown
print("\nDimension contributions:")
dims = [
    ("MRR growth",  _scale(metrics_dict.get("mrr_growth_rate", 0), -0.02, 0.08) * 20, 20),
    ("NRR",         _scale(metrics_dict.get("nrr", 1.0), 0.80, 1.30) * 20, 20),
    ("Churn",       (1 - _scale(metrics_dict.get("logo_churn_rate", 0.02), 0.0, 0.08)) * 15, 15),
    ("NPS",         _scale(metrics_dict.get("nps", 50), -20, 80) * 15, 15),
    ("Rule of 40",  _scale(metrics_dict.get("rule_of_40", 30), 10, 60) * 15, 15),
    ("LTV:CAC",     _scale(metrics_dict.get("ltv_cac_ratio", 3.0), 1.0, 6.0) * 15, 15),
]
for name, pts, weight in dims:
    print(f"  {name:<14} {pts:>5.1f} pts / {weight} pts  ({pts/weight*100:.0f}% of max)")